In [ ]:
import os

import geopandas as gpd
import pandas as pd

# External Python files
from stage_1.facade_extraction import FacadeExtractor
from stage_1.src.api_request import get_pano_id

RD_EPSG = 28992

# API options
MISSION_YEAR = 2025  # API tag
SEARCH_RADIUS = 10  # Building search radius from an initial panoramic image location

# Texture options
TEXTURE_RESOLUTION = 50
CAMERA_HEIGHT = 2  # Height of the camera mounted on the car

# Validation options before rectification (strict settings)
MIN_DISTANCE = 1.7  # Min distance camera to building
MAX_DISTANCE = 9
MIN_BUILDING_AREA = 15
MAX_BUILDING_AREA = 400
MIN_FACADE_LENGTH = 1.0

In [ ]:
# Load target areas geopackage
sample_gdf = gpd.read_file("../datasets/experiments/gevelgroen/steekproef_gebieden_train_test.gpkg")

idx = 0  # Select an area

target_area = sample_gdf.loc[idx, "geometry"]
target_area_name = sample_gdf.loc[idx, "omschrijving"]

(xmin, ymin, xmax, ymax) = target_area.bounds
BBOX_REGION = [xmin, ymax, xmax, ymin]

IMAGES_OUTPUT = f"texture_output_{target_area_name}/"

In [ ]:
# Create a directory, first check if it already exists
if not os.path.exists(IMAGES_OUTPUT):
    os.makedirs(IMAGES_OUTPUT)
else:
    print("The 'texture_output' folder already exists.")

# Get the panoramas (from API)
pano_ids = get_pano_id(MISSION_YEAR, BBOX_REGION, target_area=target_area)

print(f"Found {len(pano_ids)} panorama images")

In [ ]:
# Step 1: Identify visible building edges in range of a panoramic image

facade_extractor = FacadeExtractor(
    texture_output_folder=IMAGES_OUTPUT,
    camera_height=CAMERA_HEIGHT,
    texture_resolution=TEXTURE_RESOLUTION,
    min_building_area=MIN_BUILDING_AREA,
    max_building_area=MAX_BUILDING_AREA,
    min_building_distance=MIN_DISTANCE,
    max_building_distance=MAX_DISTANCE,
    min_facade_length=MIN_FACADE_LENGTH,
    building_search_radius=SEARCH_RADIUS,
)

facade_data = list()

facade_extractor.get_visible_building_edges(pano_ids, facade_data)

# Create pandas dataframe to easily manipulate the data
df = pd.DataFrame(facade_data)

In [ ]:
# Step 2: Drop duplicate facade textures

# Drop duplicate facade rows, keep the ones with shortest distance.
df_dedup = (
    df.sort_values("distance", ascending=True)
    .drop_duplicates(subset=["visible_point_one", "visible_point_two"])
    .sort_index()
    .reset_index(drop=True)
)

In [ ]:
# Store intermediate output
df_dedup.to_csv(f"output_{target_area_name}.df", index=False)

In [ ]:
# Re-load intermediate output (if needed)

from ast import literal_eval

def clean(x):
    return literal_eval(x)

df_dedup = pd.read_csv(f"output_{target_area_name}.df", converters={"visible_point_one": clean, "visible_point_two": clean, "observer": clean})

In [ ]:
# Step 3: Extract facade textures from panoramic images

facade_extractor.extract_facade_texture(df_dedup)

In [ ]:
# Convert output to geodataframe

import shapely.geometry as sg

RD_EPSG = 28992

segments = [
    sg.LineString((sg.Point(row.visible_point_one), sg.Point(row.visible_point_two)))
    for row in df_dedup.itertuples()
]

observers = [
    sg.Point(row.observer)
    for row in df_dedup.itertuples()
]

gdf_output = (
    gpd.GeoDataFrame(df_dedup, geometry=segments, crs=RD_EPSG)
    .drop(columns=["visible_point_one", "visible_point_two", "distance", "pano_image_url"])
    .rename(columns={"pand_id": "bag_id", "texture_height_irl": "building_height", "observer": "pano_location"})
)
gdf_output["pano_location"] = observers

In [ ]:
# Store geopackage

gdf_output.to_file(f"output_{target_area_name}.gpkg", driver="GPKG", index=False)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

from xyzservices import TileProvider

ams_tile_provider = TileProvider(
    name="Topografie, standaard visualisatie (WM)",
    url="https://t1.data.amsterdam.nl/topo_wm/{z}/{x}/{y}.png",
    attribution="data.amsterdam.nl",
)

# Generate alternating colormap by concatenating tab20
cmaps = []
idx = 0

while idx < len(gdf_output["bag_id"].unique()):
    cmaps.append(plt.get_cmap("tab20").colors)
    idx += 20

my_colors = np.vstack(cmaps)
my_cmap = mcolors.ListedColormap(my_colors)

In [ ]:
# Draw map of geopackage

import folium

map = gdf_output.explore(
    name="Facades",
    column="bag_id", 
    categorical=True,
    cmap=my_cmap,
    tiles=ams_tile_provider, 
    style_kwds={"weight": 4}, 
    popup=True, 
    popup_kwds={"localize": False},
    legend=False,
)
map = gdf_output.set_geometry("pano_location", crs=RD_EPSG).explore(
    name="Panoramas",
    m=map,
    color="red",
    tooltip="pano_id"
)
folium.LayerControl().add_to(map)

map

In [ ]:
# Save HTML map
map.save(f"output_{target_area_name}.html")